# How to Use the Saved Transformer Model for Inference
Follow these steps to load the model and make predictions on new data.

In [1]:
# ============ REPRODUCIBILITY ============
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [6]:
# ============ LOAD MODEL DEFINITION ============

class FeatureTokenizer(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.value_projection = nn.Linear(1, d_model)
        self.feature_embedding = nn.Parameter(torch.randn(num_features, d_model))

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.value_projection(x)
        x = x + self.feature_embedding
        return x

class TabularTransformer(nn.Module):
    def __init__(self, num_features, num_classes, d_model, n_heads, depth, dropout):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = self.tokenizer(x)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.transformer(x)
        cls_output = x[:, 0]
        logits = self.classifier(cls_output)
        return logits

print('Model architecture defined')

Model architecture defined


In [3]:
# ============ LOAD TRAINED MODEL ============

model_dir = Path('best_model_high_precision')

# Load config
with open(model_dir / 'reproducibility_config.json', 'r') as f:
    config = json.load(f)

print('Model Configuration:')
print(json.dumps(config, indent=2))

Model Configuration:
{
  "RANDOM_SEED": 42,
  "PYTORCH_DETERMINISTIC": true,
  "CUDNN_BENCHMARK": false,
  "MODEL_ARCHITECTURE": {
    "num_features": 58,
    "num_classes": 6,
    "d_model": 64,
    "n_heads": 4,
    "depth": 2,
    "dropout": 0.3
  },
  "TRAINING_CONFIG": {
    "batch_size": 256,
    "lr": 0.0001,
    "weight_decay": 0.0001,
    "epochs": 20,
    "optimizer": "AdamW",
    "loss_fn": "CrossEntropyLoss",
    "label_smoothing": 0.1
  },
  "PREPROCESSING": {
    "train_test_split": 0.8,
    "scaler_type": "StandardScaler",
    "scaler_fit_on": "train_only",
    "removed_features": "zero_variance_features",
    "smote_enabled": true,
    "smote_k_neighbors": 5
  },
  "PRIOR_ADJUSTMENT": {
    "enabled": true,
    "alpha": 0.8,
    "source_column": "filtered_total"
  },
  "LABEL_MAPPING": {
    "current_to_original": {
      "0": 0,
      "1": 2,
      "2": 3,
      "3": 4,
      "4": 5,
      "5": 6
    },
    "original_to_current": {
      "0": 0,
      "2": 1,
      "3"

In [7]:
# ============ INITIALIZE MODEL ============

num_features = config['MODEL_ARCHITECTURE']['num_features']
num_classes = config['MODEL_ARCHITECTURE']['num_classes']
d_model = config['MODEL_ARCHITECTURE']['d_model']
n_heads = config['MODEL_ARCHITECTURE']['n_heads']
depth = config['MODEL_ARCHITECTURE']['depth']
dropout = config['MODEL_ARCHITECTURE']['dropout']

model = TabularTransformer(
    num_features=num_features,
    num_classes=num_classes,
    d_model=d_model,
    n_heads=n_heads,
    depth=depth,
    dropout=dropout
)

# Load weights
model.load_state_dict(torch.load(model_dir / 'model_weights.pt', map_location=device))
model = model.to(device)
model.eval()

print(f'✓ Model loaded successfully')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

✓ Model loaded successfully
Total parameters: 122,182


In [10]:
# ============ LOAD SCALER & LABEL MAPPING ============

scaler = joblib.load(model_dir / 'scaler.joblib')

with open(model_dir / 'label_mapping.json', 'r') as f:
    label_data = json.load(f)

attack_names = label_data['attack_names']  # Class ID -> Attack name
current_to_original = label_data['current_to_original']
original_to_current = label_data['original_to_current']

with open(model_dir / 'features.json', 'r') as f:
    feature_names = json.load(f)

print('✓ Scaler loaded')
print('✓ Label mappings loaded')
print(f'✓ {len(feature_names)} features')
print(f'\nClass Mappings:')
for k, v in attack_names.items():
    print(f'  {k} -> {v}')

✓ Scaler loaded
✓ Label mappings loaded
✓ 58 features

Class Mappings:
  0 -> Benign
  1 -> DDoS
  2 -> DoS GoldenEye
  3 -> DoS Hulk
  4 -> DoS Slowhttptest
  5 -> DoS slowloris


In [11]:
# ============ TEST WITH SAMPLE DATA ============

# Generate random test data (58 features)
X_test_sample = np.random.randn(100, num_features).astype(np.float32)

# Scale using the trained scaler
X_test_scaled = scaler.transform(X_test_sample)

# Convert to tensor
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

# Run inference
with torch.no_grad():
    predictions = model(X_test_tensor)
    pred_labels = torch.argmax(predictions, dim=1).cpu().numpy()

# Convert to attack names
pred_names = [attack_names[str(int(label))] for label in pred_labels]

print('Sample Predictions (first 10):')
for i in range(min(10, len(pred_names))):
    print(f'  Sample {i+1}: {pred_names[i]} (class {pred_labels[i]})')

print(f'\n✓ Inference working correctly')

Sample Predictions (first 10):
  Sample 1: DoS GoldenEye (class 2)
  Sample 2: DoS GoldenEye (class 2)
  Sample 3: DoS GoldenEye (class 2)
  Sample 4: DoS GoldenEye (class 2)
  Sample 5: DoS GoldenEye (class 2)
  Sample 6: DoS GoldenEye (class 2)
  Sample 7: DoS GoldenEye (class 2)
  Sample 8: DoS GoldenEye (class 2)
  Sample 9: DoS GoldenEye (class 2)
  Sample 10: DoS GoldenEye (class 2)

✓ Inference working correctly


In [24]:
# ============ TEST REPRODUCIBILITY ============
# This step requires df_cleaned.csv 
# Download from: https://drive.google.com/file/d/1oHnlFd44RHHOPud9DoWAEtxi7cMrZMNc/view?usp=sharing

csv_path = Path('df_cleaned.csv')

if csv_path.exists():
    print('Found df_cleaned.csv - Running reproducibility check...\n')
    
    # Load dataset
    df = pd.read_csv(csv_path)
    print(f'Dataset shape: {df.shape}')
    print(f'Unique labels in raw data: {sorted(df.iloc[:, -1].unique())}')
    
    # CRITICAL: Filter to keep only the 6 classes the model was trained on
    # Original labels 1, 7, 10, 11, 12 were removed during training
    keep_labels = [0, 2, 3, 4, 5, 6]  # Classes to keep
    df_filtered = df[df.iloc[:, -1].isin(keep_labels)].copy().reset_index(drop=True)
    print(f'After filtering to keep labels {keep_labels}: {df_filtered.shape}')
    
    # Separate features and labels
    X = df_filtered.iloc[:, :-1].values
    y_original = df_filtered.iloc[:, -1].values
    
    # CRITICAL: Remap original labels to model labels
    # Original: 0, 2, 3, 4, 5, 6 → Model: 0, 1, 2, 3, 4, 5
    original_to_current = {0: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5}
    y = np.array([original_to_current[int(label)] for label in y_original])
    
    print(f'\nFiltered data shape: {X.shape}')
    print(f'Labels shape: {y.shape}')
    print(f'Original labels: {np.unique(y_original)}')
    print(f'Remapped labels: {np.unique(y)}')
    print(f'Class distribution:')
    for orig, curr in sorted(original_to_current.items()):
        count = np.sum(y == curr)
        attack = attack_names[str(curr)]
        print(f'  {attack:20s} (orig={orig}, curr={curr}): {count:10d}')
else:
    print('⚠️ df_cleaned.csv not found')
    print('Download from: https://drive.google.com/file/d/1oHnlFd44RHHOPud9DoWAEtxi7cMrZMNc/view?usp=sharing')
    print('Place it in the same folder as this notebook')

Found df_cleaned.csv - Running reproducibility check...

Dataset shape: (2233159, 59)
Unique labels in raw data: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]
After filtering to keep labels [0, 2, 3, 4, 5, 6]: (2218426, 59)

Filtered data shape: (2218426, 58)
Labels shape: (2218426,)
Original labels: [0 2 3 4 5 6]
Remapped labels: [0 1 2 3 4 5]
Class distribution:
  Benign               (orig=0, curr=0):    1896667
  DDoS                 (orig=2, curr=1):     128014
  DoS GoldenEye        (orig=3, curr=2):      10286
  DoS Hulk             (orig=4, curr=3):     172846
  DoS Slowhttptest     (orig=5, curr=4):       5228
  DoS slowloris        (orig=6, curr=5):       5385


In [ ]:
# ============ TRAIN-TEST SPLIT & INFERENCE ============

if csv_path.exists():
    # Recreate train-test split (same as training)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    print(f'Train set: {X_train.shape}')
    print(f'Test set: {X_test.shape}\n')
    
    # Scale test data
    X_test_scaled = scaler.transform(X_test)
    
    # ============ PRIOR ADJUSTMENT ============
    print(f'Calculating prior adjustment...')
    
    filtered_counts = {0: 1896667, 1: 128014, 2: 10286, 3: 172846, 4: 5228, 5: 5385}
    counts = np.array([filtered_counts[i] for i in range(num_classes)]) + 1.0
    priors = counts / counts.sum()
    alpha = 0.8  # From config
    adjustment = alpha * np.log(priors)
    adjustment_tensor = torch.tensor(adjustment, dtype=torch.float32, device=device)
    
    print(f'Prior adjustment ready\n')
    
    # Run inference
    batch_size = 5000
    pred_labels = []
    
    print(f'Running inference...')
    for i in range(0, len(X_test_scaled), batch_size):
        X_batch = X_test_scaled[i:i+batch_size]
        X_batch_tensor = torch.tensor(X_batch, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            batch_predictions = model(X_batch_tensor)
            adjusted_predictions = batch_predictions + adjustment_tensor
            batch_pred_labels = torch.argmax(adjusted_predictions, dim=1).cpu().numpy()
            pred_labels.extend(batch_pred_labels)
    
    pred_labels = np.array(pred_labels)
    
    # ============ EVALUATE ============
    accuracy = accuracy_score(y_test, pred_labels)
    precision = precision_score(y_test, pred_labels, average='weighted', zero_division=0)
    recall = recall_score(y_test, pred_labels, average='weighted', zero_division=0)
    f1 = f1_score(y_test, pred_labels, average='weighted', zero_division=0)
    
    print(f'\n' + '='*60)
    print('TEST SET RESULTS')
    print('='*60)
    print(f'Accuracy:  {accuracy:.4f} (94.52% expected)')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1-Score:  {f1:.4f}')
    print('='*60)
    
    # ============ CLASS-WISE PERFORMANCE ============
    from sklearn.metrics import precision_recall_fscore_support
    
    print(f"\n" + "="*90)
    print("PERFORMANCE BY CLASS")
    print("="*90)
    
    precision_list, recall_list, f1_list, support_list = precision_recall_fscore_support(
        y_test, pred_labels, zero_division=0
    )
    
    print(f"\n{'Class':<25} {'Samples':<10} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
    print("-"*90)
    
    for class_idx in range(num_classes):
        class_name = attack_names[str(class_idx)]
        class_count = int(support_list[class_idx])
        
        if class_count > 0:
            tp = np.sum((y_test == class_idx) & (pred_labels == class_idx))
            class_accuracy = tp / class_count
            
            print(f"{class_name:<25} {class_count:<10d} {class_accuracy:<12.4f} "
                  f"{float(precision_list[class_idx]):<12.4f} {float(recall_list[class_idx]):<12.4f} "
                  f"{float(f1_list[class_idx]):<12.4f}")

else:
    print('⚠️ df_cleaned.csv not found')


RE-EVALUATING WITH PRIOR ADJUSTMENT (Correct Method)

Class Frequency Adjustment (alpha=0.8):
  Class 0 (Benign              ): count= 1896667, adjustment= -0.1254
  Class 1 (DDoS                ): count=  128014, adjustment= -2.2819
  Class 2 (DoS GoldenEye       ): count=   10286, adjustment= -4.2989
  Class 3 (DoS Hulk            ): count=  172846, adjustment= -2.0417
  Class 4 (DoS Slowhttptest    ): count=    5228, adjustment= -4.8403
  Class 5 (DoS slowloris       ): count=    5385, adjustment= -4.8166

Adjustment tensor shape: torch.Size([6])
Adjustment values: tensor([-0.1254, -2.2819, -4.2989, -2.0417, -4.8403, -4.8166], device='cuda:0')

Running inference with prior adjustment...

COMPARISON: WITHOUT vs WITH PRIOR ADJUSTMENT

Metric               Without Adjustment        With Adjustment          
----------------------------------------------------------------------
Accuracy             0.8485                    0.9452                   
Precision            0.9545         